# RAG Pipeline using LangChain, FAISS and Google Gemini

## What this notebook does

This notebook builds a complete, minimal Retrieval Augmented Generation (RAG) pipeline over a single text document (Paul Graham's essay "What I Worked On"). It answers questions about the essay by finding the most relevant passages first, then asking a large language model to write an answer using only those passages.

## The three stages of a RAG pipeline

Every RAG system, no matter how complex, is built from the same three stages. This notebook is organized around them directly, one section per stage.

1. **Data Ingestion**: read the raw source document, split it into small overlapping chunks, convert each chunk into a numeric vector (an embedding), and store all of the vectors in a searchable index.
2. **Data Retrieval**: given a user's question, search the index for the chunks whose vectors are closest to the question's own vector. These are the passages most likely to contain the answer.
3. **Data Generation**: hand the retrieved passages to a large language model as context, together with the original question, and let the model write a grounded, accurate answer instead of relying only on what it already knows.

## Tools used

* **LangChain**: provides the building blocks used to load documents, split text, wrap a vector store as a retriever, build prompts, and chain everything together.
* **FAISS**: Facebook AI Similarity Search, a fast, local, in memory vector index used to store and search the document chunk embeddings. No external database or network service is required.
* **Hugging Face Sentence Transformers**: a small, free, local embedding model (`all-MiniLM-L6-v2`) used to turn text into vectors.
* **Google Gemini**: the large language model that reads the retrieved passages and generates the final answer, accessed through `langchain-google-genai`.

The notebook runs top to bottom in order. Each section explains what is happening and why before showing the code.

# Stage 1: Data Ingestion

Data ingestion is the setup work that only needs to happen once (or whenever the source data changes). It takes a raw document and turns it into a searchable index. This stage has four parts:

1. Install the packages needed to load and process documents.
2. Load the raw text file into memory.
3. Split the text into small, overlapping chunks, since feeding an entire essay to an embedding model or an LLM at once is inefficient and often exceeds size limits.
4. Convert every chunk into a vector and store the vectors in a FAISS index.

## Step 1.1: Install the ingestion packages

* `langchain` and `langchain-community` provide the document loaders and text splitters.
* `openai` and `tiktoken` are common LangChain dependencies pulled in for compatibility with LangChain's broader tooling.
* `rapidocr-onnxruntime` enables optical character recognition for loaders that need to read text out of images or scanned PDFs.
* `python-dotenv` loads secrets like API keys from a local `.env` file instead of hardcoding them in the notebook.

In [ ]:
# Install the core packages needed for loading and processing documents.
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

## Step 1.2: Load the Google API key safely

The Gemini model used later in this notebook needs a `GOOGLE_API_KEY`. Instead of typing the key directly into a cell, where it could accidentally be committed to GitHub, it is loaded from a local `.env` file using `python-dotenv`.

To use this notebook, create a file named `.env` in the same folder, and add a single line to it:

```
GOOGLE_API_KEY=your-key-here
```

Then add `.env` to `.gitignore` so it is never committed. If the key is not found in the environment after `load_dotenv()` runs, a hidden prompt asks for it instead, so the value is never printed to the screen.

In [ ]:
import os
import getpass
from dotenv import load_dotenv

# Load any values found in a local .env file into the environment.
load_dotenv()

# Make sure GOOGLE_API_KEY is available. If it was not found in .env,
# ask for it interactively without echoing it to the screen.
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Provide your GOOGLE_API_KEY: ")
else:
    os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

print("GOOGLE_API_KEY is loaded into the environment.")

## Step 1.3: Load the source document

`TextLoader` reads a plain text file from disk and wraps it in LangChain's `Document` object, which stores the file's text alongside a small metadata dictionary (such as the file path).

Update `DATA_FILE_PATH` below to point at wherever the source `.txt` file actually lives on the current machine. It is kept as a single variable at the top of the cell so it is easy to change without hunting through the rest of the notebook.

Note: `langchain_community` currently prints a deprecation warning for some of its loaders, including `TextLoader`, since the LangChain project is gradually moving integrations into their own standalone packages. The loader still works correctly today, but keep an eye on LangChain's migration guides for when a dedicated replacement package becomes the recommended path.

In [ ]:
from langchain_community.document_loaders import TextLoader

# Update this path to point at the essay (or any other .txt file) to ingest.
DATA_FILE_PATH = r"F:\AI Projects\End-to-End-LLMOPs-Project\data\paul_graham_essay.txt"

loader = TextLoader(DATA_FILE_PATH, encoding="utf8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")

In [ ]:
# Preview the first 500 characters of the loaded document to confirm it read correctly.
documents[0].page_content[:500]

## Step 1.4: Split the document into chunks

Large language models and embedding models both work best with short, focused pieces of text rather than an entire essay at once. `RecursiveCharacterTextSplitter` breaks the document into overlapping chunks.

* `chunk_size=200` limits each chunk to roughly 200 characters.
* `chunk_overlap=20` lets consecutive chunks share 20 characters at the boundary, so a sentence that gets cut in half at a chunk edge still appears in full inside at least one chunk.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
text_chunks = text_splitter.split_documents(documents)

print(f"Split the document into {len(text_chunks)} chunks.")

## Step 1.5: Install the embedding and vector store packages

* `faiss-cpu` is the CPU only build of FAISS, the vector index used to store and search chunk embeddings.
* `langchain-huggingface` provides LangChain's wrapper around Hugging Face embedding models.
* `sentence-transformers` is the underlying library that actually runs the embedding model.

In [ ]:
! uv pip install faiss-cpu langchain-huggingface sentence-transformers

## Step 1.6: Embed every chunk

`all-MiniLM-L6-v2` is a small, fast, freely available sentence embedding model from Hugging Face. It converts each text chunk into a fixed length numeric vector that captures its meaning, so that chunks with similar meaning end up with vectors that are close together in that vector space.

This model runs locally on this machine (no external API call or key is required for the embedding step itself).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(embedding_model)

## Step 1.7: Build the FAISS vector index

`FAISS.from_documents(...)` does two things in one call: it runs every chunk through the embedding model, and it stores the resulting vectors inside a FAISS index that supports fast nearest neighbor search. This index is the searchable knowledge base that the retrieval stage will query.

Since FAISS here is used purely in memory, this index exists only for the current notebook session. For a project that needs to reuse the same index across sessions, FAISS supports saving the index to disk with `vectorstore.save_local(...)` and reloading it later with `FAISS.load_local(...)`.

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(text_chunks, embedding_model)
vectorstore

Data ingestion is now complete. The essay has been loaded, split into 487 small overlapping chunks, embedded, and stored inside a searchable FAISS index. The next stage uses this index to answer real questions.

# Stage 2: Data Retrieval

Retrieval is the step that runs every time a user asks a question. Given a question, the system searches the FAISS index built during ingestion and pulls out the chunks that are most semantically similar to that question. Those chunks become the context that the generation stage will use to answer accurately.

## Step 2.1: Wrap the vector store as a retriever

`vectorstore.as_retriever()` turns the FAISS index into a standardized LangChain retriever object. A retriever exposes a simple, consistent interface (give it a query, get back a list of relevant documents) that can be swapped in and out of larger chains without changing the rest of the pipeline. By default it returns the 4 most similar chunks for any given query.

In [ ]:
retriever = vectorstore.as_retriever()

## Step 2.2: Test retrieval directly

Before wiring retrieval into a full chain, it helps to see it working on its own. `vectorstore.similarity_search(query, k=4)` runs the same underlying search that the retriever will use, and returns the top `k` closest chunks by meaning, not by keyword matching. Notice how the retrieved passages below relate to the question even when they do not repeat its exact wording, which is the benefit of searching by embedding similarity rather than plain text search.

In [ ]:
# Perform a similarity search directly against the vector store.
query = "What was the first computer Paul Graham programmed on?"
docs = vectorstore.similarity_search(query, k=4)

# Display the retrieved chunks.
for i, doc in enumerate(docs):
    print(f"Document {i + 1}:")
    print(doc.page_content)
    print("=" * 50)

Retrieval is now confirmed to work. The next stage builds the prompt and the language model that will turn these retrieved chunks into a written answer.

# Stage 3: Data Generation

Generation is where a large language model reads the retrieved context and the user's question, and writes the final answer. This stage has four parts: define the prompt template, define how the model's raw output should be parsed, load the language model, and chain everything together into a single, runnable pipeline.

## Step 3.1: Define the prompt template

The prompt template is the instruction sheet handed to the language model on every single call. It has two placeholders, `{question}` and `{context}`, that get filled in automatically at run time. The instructions explicitly tell the model to rely on the retrieved context, to admit when it does not know the answer instead of guessing, and to keep the answer short. This is what keeps the model's answers grounded in the actual essay instead of drifting into unrelated general knowledge.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question and answer tasks.
Use the following pieces of retrieved context to answer the question.
If you do not know the answer, just say that you do not know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

## Step 3.2: Define the output parser

A raw LangChain model response is a structured message object, not a plain string. `StrOutputParser` extracts just the text content of that response, which is the format needed for a clean, readable final answer.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

## Step 3.3: Load the Gemini language model

`ChatGoogleGenerativeAI` connects to Google's Gemini model through LangChain, using the `GOOGLE_API_KEY` loaded during the ingestion stage. `temperature=0` makes the model as deterministic and focused as possible, which is usually the right setting for a question and answer task where consistent, factual answers matter more than creative variation.

Note: model names and version availability change over time as Google releases new Gemini models and retires older ones. If the model name below returns a "not found" style error, check Google AI Studio or Vertex AI for the current list of available Gemini model ids and update the `model` argument accordingly.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

## Step 3.4: Chain retrieval, prompting and generation together

This is where the three stages of the pipeline meet. LangChain's pipe syntax (`|`) reads like a flow diagram, left to right:

1. `{"context": retriever, "question": RunnablePassthrough()}` takes the incoming question, sends it to the retriever to fetch relevant chunks (becoming `context`), and also passes the original question straight through unchanged (becoming `question`).
2. `prompt` fills the template from Step 3.1 with those two values.
3. `llm` sends the filled in prompt to Gemini and gets back a response.
4. `output_parser` extracts the plain text answer from that response.

The result, `rag_chain`, is a single object that can be called with just a question and will run the entire ingestion informed pipeline end to end.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

## Step 3.5: Ask questions

With the full chain assembled, answering a question is a single `.invoke(...)` call. Behind the scenes, this triggers retrieval against the FAISS index, fills the prompt with the retrieved passages, calls Gemini, and parses the response, all in one step.

In [ ]:
rag_chain.invoke("What inspired the creation of Y Combinator?")

In [ ]:
rag_chain.invoke("Which programming language did he learn for AI?")

## Step 3.6: Print a formatted question and answer report

This final cell wraps the same chain call in a small, readable print block, useful for logging or demoing a single question and answer pair clearly.

In [ ]:
query = "Which programming language did he learn for AI?"

response = rag_chain.invoke(query)

print("=" * 60)
print("Question:")
print(query)
print("\nAnswer:")
print(response)
print("=" * 60)

## Conclusion

This notebook built a complete RAG pipeline in three clear stages:

1. **Data Ingestion**: the essay was loaded, split into 487 overlapping chunks, embedded with a local Hugging Face model, and stored in a FAISS vector index.
2. **Data Retrieval**: a retriever was built on top of the FAISS index, capable of pulling out the most semantically relevant chunks for any question.
3. **Data Generation**: a prompt template, an output parser, and the Gemini language model were chained together with the retriever, producing a single callable pipeline that answers questions using only the retrieved passages as grounding.

### Where to go from here

* Increase or decrease `chunk_size` and `chunk_overlap` in Step 1.4 and observe how it changes retrieval quality and answer accuracy.
* Try a larger `k` value in the retriever from Step 2.1 (for example `vectorstore.as_retriever(search_kwargs={"k": 6})`) to give the model more candidate passages per question.
* Save the FAISS index to disk with `vectorstore.save_local(...)` so the ingestion stage does not need to rerun every time the notebook is restarted.
* Swap in a different document loader (for example a PDF or web page loader) to run this same pipeline over a different kind of source document.
* Add a simple evaluation step that checks generated answers against a small set of known question and answer pairs to track pipeline quality over time.